# Assistente Médico com LangChain

## Objetivo
Este notebook demonstra a construção de um **assistente médico contextualizado** utilizando **LangChain** e um modelo de linguagem ajustado na etapa anterior do projeto.

A proposta desta etapa é mostrar como a LLM pode ser utilizada como núcleo de um assistente clínico capaz de:

- interpretar informações de um paciente;
- responder dúvidas clínicas com base no contexto fornecido;
- sugerir próximos passos de forma segura;
- produzir respostas estruturadas para posterior integração com fluxos automatizados.

## Observação importante
Este notebook utiliza **dados simulados/sintéticos**, sem qualquer dado real de paciente, o que é adequado para fins acadêmicos e demonstração técnica.

## 1. Bibliotecas e configuração inicial

Nesta seção, importamos as bibliotecas necessárias e configuramos o ambiente.  
O notebook foi preparado para funcionar em dois modos:

1. **Modo Azure OpenAI**: usa variáveis de ambiente, caso estejam disponíveis;
2. **Modo demonstração**: usa uma implementação local simulada para permitir a execução mesmo sem credenciais.

In [1]:

import os
import json
from datetime import datetime
from typing import Dict, Any, List, TypedDict

# LangChain core
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda

USE_AZURE = all(
    os.getenv(var)
    for var in ["AZURE_OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_DEPLOYMENT_NAME"]
)

llm = None
execution_mode = "demonstracao"

if USE_AZURE:
    try:
        from langchain_openai import AzureChatOpenAI

        llm = AzureChatOpenAI(
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21"),
            temperature=0.2,
        )
        execution_mode = "azure_openai"
    except Exception as e:
        print(f"Falha ao inicializar AzureChatOpenAI. Usando modo demonstração. Erro: {e}")
        llm = None

print("Modo de execução:", execution_mode)

Modo de execução: azure_openai


## 2. Definição do caso clínico simulado

Para demonstrar o assistente, utilizamos um caso clínico estruturado contendo:

- identificação simulada do paciente;
- sintomas;
- sinais vitais;
- histórico resumido;
- exames pendentes;
- observações clínicas.

Esse formato é intencionalmente próximo de dados estruturados de prontuário, ainda que aqui esteja representado em memória.

In [2]:

paciente = {
    "paciente_id": "PAC-001",
    "idade": 58,
    "sexo": "Masculino",
    "sintomas": ["dor torácica", "dispneia leve", "fadiga"],
    "sinais_vitais": {
        "pressao_arterial": "150/95 mmHg",
        "frequencia_cardiaca": "102 bpm",
        "temperatura": "36.8 C",
        "saturacao_oxigenio": "95%"
    },
    "historico": [
        "hipertensão arterial sistêmica",
        "tabagismo prévio",
        "histórico familiar de doença cardiovascular"
    ],
    "exames_pendentes": ["troponina", "eletrocardiograma", "hemograma"],
    "observacoes_clinicas": "Paciente refere início dos sintomas nas últimas 6 horas."
}

pergunta_clinica = (
    "Com base nos dados informados, quais hipóteses devem ser consideradas e "
    "quais próximos passos assistenciais podem ser sugeridos?"
)

print(json.dumps(paciente, ensure_ascii=False, indent=2))
print("\nPergunta clínica:", pergunta_clinica)

{
  "paciente_id": "PAC-001",
  "idade": 58,
  "sexo": "Masculino",
  "sintomas": [
    "dor torácica",
    "dispneia leve",
    "fadiga"
  ],
  "sinais_vitais": {
    "pressao_arterial": "150/95 mmHg",
    "frequencia_cardiaca": "102 bpm",
    "temperatura": "36.8 C",
    "saturacao_oxigenio": "95%"
  },
  "historico": [
    "hipertensão arterial sistêmica",
    "tabagismo prévio",
    "histórico familiar de doença cardiovascular"
  ],
  "exames_pendentes": [
    "troponina",
    "eletrocardiograma",
    "hemograma"
  ],
  "observacoes_clinicas": "Paciente refere início dos sintomas nas últimas 6 horas."
}

Pergunta clínica: Com base nos dados informados, quais hipóteses devem ser consideradas e quais próximos passos assistenciais podem ser sugeridos?


## 3. Funções auxiliares

Aqui definimos funções de apoio para:

- registrar logs de execução;
- preparar contexto estruturado para o prompt;
- simular resposta da LLM quando a infraestrutura externa não estiver disponível.

Essas funções facilitam a rastreabilidade da demonstração.

In [6]:
LOG_FILE = os.path.join("B:/Pós/hospital-ai-diagnosis/notebooks/Fase 3/04_logs", "assistente_langchain_logs.jsonl")

def registrar_log(etapa: str, entrada: Dict[str, Any], saida: Dict[str, Any]) -> None:
    evento = {
        "timestamp": datetime.utcnow().isoformat(),
        "etapa": etapa,
        "entrada": entrada,
        "saida": saida
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(evento, ensure_ascii=False) + "\n")

def montar_contexto_paciente(paciente: Dict[str, Any]) -> str:
    return json.dumps(paciente, ensure_ascii=False, indent=2)

def resposta_simulada(entrada: Dict[str, Any]) -> Dict[str, Any]:
    dados = entrada["paciente"]
    sintomas = [s.lower() for s in dados.get("sintomas", [])]
    exames = dados.get("exames_pendentes", [])

    hipoteses = []
    if "dor torácica" in sintomas:
        hipoteses.append("síndrome coronariana aguda")
    if "dispneia leve" in sintomas:
        hipoteses.append("insuficiência cardíaca inicial ou comprometimento cardiopulmonar")
    if not hipoteses:
        hipoteses.append("necessidade de investigação clínica complementar")

    recomendacoes = [
        "priorizar avaliação médica presencial",
        "acompanhar sinais vitais continuamente",
        "executar exames pendentes para melhor definição diagnóstica"
    ]

    if "eletrocardiograma" in exames:
        recomendacoes.append("realizar eletrocardiograma com prioridade")
    if "troponina" in exames:
        recomendacoes.append("realizar dosagem de troponina para investigação de lesão miocárdica")

    return {
        "resumo_clinico": "Paciente com quadro potencialmente cardiovascular que requer investigação prioritária.",
        "hipoteses_principais": hipoteses,
        "proximos_passos": recomendacoes,
        "restricao_seguranca": (
            "O assistente não realiza prescrição medicamentosa nem substitui avaliação médica."
        ),
        "justificativa": (
            "As sugestões foram geradas com base nos sintomas relatados, sinais vitais e histórico clínico fornecidos."
        ),
        "baseado_em": {
            "sintomas": dados.get("sintomas", []),
            "sinais_vitais": dados.get("sinais_vitais", {}),
            "historico": dados.get("historico", []),
            "exames_pendentes": exames
        }
    }

## 4. Prompt do assistente médico

O prompt abaixo orienta a LLM para:

- responder de forma objetiva e técnica;
- considerar o contexto completo do paciente;
- não prescrever diretamente medicamentos;
- sugerir próximos passos com linguagem segura;
- retornar a resposta em **JSON estruturado**, facilitando auditoria e integração posterior.

Esse formato também favorece o reaproveitamento do assistente em fluxos com LangGraph.

In [7]:

system_prompt = '''
Você é um assistente virtual médico de apoio à decisão clínica.
Sua função é auxiliar profissionais de saúde com base no contexto do paciente informado.

Regras obrigatórias:
1. Não prescreva medicamentos diretamente.
2. Não substitua a avaliação médica humana.
3. Sugira hipóteses e próximos passos assistenciais com cautela.
4. Seja claro, técnico e objetivo.
5. Explique brevemente em que dados a resposta se baseou.
6. Responda APENAS em JSON válido.

Estrutura obrigatória da resposta:
{
  "resumo_clinico": "...",
  "hipoteses_principais": ["..."],
  "proximos_passos": ["..."],
  "restricao_seguranca": "...",
  "justificativa": "...",
  "baseado_em": {
    "sintomas": [],
    "sinais_vitais": {},
    "historico": [],
    "exames_pendentes": []
  }
}
'''

human_prompt = '''
Contexto do paciente:
{contexto_paciente}

Pergunta clínica:
{pergunta_clinica}
'''

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", human_prompt)
])

print("Prompt configurado com sucesso.")

Prompt configurado com sucesso.


## 5. Construção da chain com LangChain

Nesta etapa montamos a chain principal.

Fluxo lógico:
1. recebimento do contexto do paciente;
2. montagem do prompt;
3. chamada da LLM;
4. parsing do JSON retornado;
5. fallback para modo demonstração, se necessário.

In [8]:

json_parser = JsonOutputParser()

def executar_chain(entrada: Dict[str, Any]) -> Dict[str, Any]:
    contexto_paciente = montar_contexto_paciente(entrada["paciente"])
    payload = {
        "contexto_paciente": contexto_paciente,
        "pergunta_clinica": entrada["pergunta_clinica"]
    }

    if llm is None:
        resultado = resposta_simulada(entrada)
        registrar_log("assistente_langchain_modo_demo", entrada, resultado)
        return resultado

    try:
        chain = prompt | llm | json_parser
        resultado = chain.invoke(payload)
        registrar_log("assistente_langchain_llm", entrada, resultado)
        return resultado
    except Exception as e:
        resultado = resposta_simulada(entrada)
        resultado["fallback_reason"] = f"Falha na chamada da LLM: {str(e)}"
        registrar_log("assistente_langchain_fallback", entrada, resultado)
        return resultado

assistente_chain = RunnableLambda(executar_chain)
print("Chain criada com sucesso.")

Chain criada com sucesso.


## 6. Execução do assistente

Agora executamos a chain com o caso clínico simulado.  
O objetivo é demonstrar que o assistente consegue:

- compreender o contexto do paciente;
- gerar hipóteses clínicas;
- sugerir próximos passos;
- registrar restrições de segurança;
- produzir justificativa estruturada.

In [9]:

entrada_assistente = {
    "paciente": paciente,
    "pergunta_clinica": pergunta_clinica
}

resultado_assistente = assistente_chain.invoke(entrada_assistente)
print(json.dumps(resultado_assistente, ensure_ascii=False, indent=2))

{
  "resumo_clinico": "Paciente com quadro potencialmente cardiovascular que requer investigação prioritária.",
  "hipoteses_principais": [
    "síndrome coronariana aguda",
    "insuficiência cardíaca inicial ou comprometimento cardiopulmonar"
  ],
  "proximos_passos": [
    "priorizar avaliação médica presencial",
    "acompanhar sinais vitais continuamente",
    "executar exames pendentes para melhor definição diagnóstica",
    "realizar eletrocardiograma com prioridade",
    "realizar dosagem de troponina para investigação de lesão miocárdica"
  ],
  "restricao_seguranca": "O assistente não realiza prescrição medicamentosa nem substitui avaliação médica.",
  "justificativa": "As sugestões foram geradas com base nos sintomas relatados, sinais vitais e histórico clínico fornecidos.",
  "baseado_em": {
    "sintomas": [
      "dor torácica",
      "dispneia leve",
      "fadiga"
    ],
    "sinais_vitais": {
      "pressao_arterial": "150/95 mmHg",
      "frequencia_cardiaca": "102 bp

C:\Users\filip\AppData\Local\Temp\ipykernel_12032\3033826890.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


## 7. Interpretação do resultado

A saída foi estruturada de forma a facilitar:

- leitura humana;
- auditoria;
- reutilização em fluxos posteriores;
- demonstração acadêmica dos mecanismos de segurança.

A seguir, destacamos os principais componentes da resposta.

In [10]:

print("Resumo clínico:")
print(resultado_assistente.get("resumo_clinico", ""))

print("\nHipóteses principais:")
for item in resultado_assistente.get("hipoteses_principais", []):
    print("-", item)

print("\nPróximos passos:")
for item in resultado_assistente.get("proximos_passos", []):
    print("-", item)

print("\nRestrição de segurança:")
print(resultado_assistente.get("restricao_seguranca", ""))

print("\nJustificativa:")
print(resultado_assistente.get("justificativa", ""))

Resumo clínico:
Paciente com quadro potencialmente cardiovascular que requer investigação prioritária.

Hipóteses principais:
- síndrome coronariana aguda
- insuficiência cardíaca inicial ou comprometimento cardiopulmonar

Próximos passos:
- priorizar avaliação médica presencial
- acompanhar sinais vitais continuamente
- executar exames pendentes para melhor definição diagnóstica
- realizar eletrocardiograma com prioridade
- realizar dosagem de troponina para investigação de lesão miocárdica

Restrição de segurança:
O assistente não realiza prescrição medicamentosa nem substitui avaliação médica.

Justificativa:
As sugestões foram geradas com base nos sintomas relatados, sinais vitais e histórico clínico fornecidos.


## 8. Explainability

A explicabilidade, neste contexto, é demonstrada por meio do campo `baseado_em`, que explicita quais dados do paciente sustentaram a recomendação produzida.

Embora ainda não haja busca em base documental externa neste notebook, a resposta já evidencia:

- sintomas considerados;
- sinais vitais analisados;
- histórico clínico utilizado;
- exames pendentes incorporados à sugestão.

In [11]:

print(json.dumps(resultado_assistente.get("baseado_em", {}), ensure_ascii=False, indent=2))

{
  "sintomas": [
    "dor torácica",
    "dispneia leve",
    "fadiga"
  ],
  "sinais_vitais": {
    "pressao_arterial": "150/95 mmHg",
    "frequencia_cardiaca": "102 bpm",
    "temperatura": "36.8 C",
    "saturacao_oxigenio": "95%"
  },
  "historico": [
    "hipertensão arterial sistêmica",
    "tabagismo prévio",
    "histórico familiar de doença cardiovascular"
  ],
  "exames_pendentes": [
    "troponina",
    "eletrocardiograma",
    "hemograma"
  ]
}


## 9. Visualização do log gerado

A rastreabilidade é importante para auditoria.  
Nesta seção, exibimos os últimos registros gravados pelo assistente durante a execução do notebook.

In [12]:

def ler_ultimos_logs(path: str, n: int = 3) -> List[Dict[str, Any]]:
    if not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        linhas = f.readlines()
    return [json.loads(linha) for linha in linhas[-n:]]

ultimos_logs = ler_ultimos_logs(LOG_FILE, n=3)
print(json.dumps(ultimos_logs, ensure_ascii=False, indent=2))

[
  {
    "timestamp": "2026-03-21T17:14:30.692302",
    "etapa": "assistente_langchain_fallback",
    "entrada": {
      "paciente": {
        "paciente_id": "PAC-001",
        "idade": 58,
        "sexo": "Masculino",
        "sintomas": [
          "dor torácica",
          "dispneia leve",
          "fadiga"
        ],
        "sinais_vitais": {
          "pressao_arterial": "150/95 mmHg",
          "frequencia_cardiaca": "102 bpm",
          "temperatura": "36.8 C",
          "saturacao_oxigenio": "95%"
        },
        "historico": [
          "hipertensão arterial sistêmica",
          "tabagismo prévio",
          "histórico familiar de doença cardiovascular"
        ],
        "exames_pendentes": [
          "troponina",
          "eletrocardiograma",
          "hemograma"
        ],
        "observacoes_clinicas": "Paciente refere início dos sintomas nas últimas 6 horas."
      },
      "pergunta_clinica": "Com base nos dados informados, quais hipóteses devem ser considerad

## 10. Conclusão

Este notebook demonstrou a implementação de um **assistente médico contextualizado com LangChain**, cobrindo os seguintes pontos da Fase 3:

- integração da LLM ao pipeline;
- contextualização da resposta com dados do paciente;
- resposta estruturada em JSON;
- restrições básicas de segurança;
- registro de logs para rastreabilidade;
- base inicial de explainability.

### Limitações atuais
- os dados do paciente são simulados;
- não há integração com prontuário real;
- a validação humana formal foi tratada no notebook de LangGraph;
- a fonte da resposta ainda é o contexto estruturado, e não uma base hospitalar documental.